# Imports

In [1]:
from _spo_utils import camel_to_snake, import_json

import pandas as pd 
import json

# Constants

In [ ]:
PATH_SPOTIFY = '../../data/2_processed/final_df.csv'
PATH_MARQUEE = '../../data/1_raw/Marquee.json'
PATH_HISTORY = '../../data/1_raw/Streaming_History_Audio_2017-2026.json'
PATH_LIBRARY = '../../data/1_raw/YourLibrary.json'
PATH_PLAYLIST = '../../data/1_raw/Playlist1.json'

<h1>JSON File Convensions & Imports</h1>

<h3 style='color:gray;'>Imports Marquee JSON file (1/4)</h3>

In [ ]:
marquee = import_json(PATH_MARQUEE)
marquee.head()

,artist_name,segment
0,DJ Guih Da ZO,Light listeners
1,Dexhenry,Previously Active Listeners
2,Vintage Culture,Previously Active Listeners
3,DJ R7,Previously Active Listeners
4,Gnarls Barkley,Previously Active Listeners


<h3 style='color:gray;'>Imports Streaming History JSON file, using record_path (2/4)</h3>

In [ ]:
# History — simple
history = import_json(PATH_HISTORY)
history.head()

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2017-11-21T11:46:01Z,"iOS 9.3.5 (iPad2,5)",6826,BR,201.6.225.101,The Society,Christopher Drake,Injustice 2: Original Video Game Soundtrack,spotify:track:74ovIDtL0HzDazMENhR0yX,None,...,None,None,None,playbtn,endplay,True,False,False,NaN,False
1,2017-11-21T11:46:18Z,"iOS 9.3.5 (iPad2,5)",16648,BR,201.6.225.101,Injustice 2 Main Theme,Christopher Drake,Injustice 2: Original Video Game Soundtrack,spotify:track:0QKHV9dERThRvpdzQWy8qd,None,...,None,None,None,clickrow,fwdbtn,False,False,False,NaN,False
2,2017-11-21T11:46:19Z,"iOS 9.3.5 (iPad2,5)",766,BR,201.6.225.101,Brainiac Takes Krypton,Christopher Drake,Injustice 2: Original Video Game Soundtrack,spotify:track:7urxa3s1BTWXmx2Eb9QWLp,None,...,None,None,None,fwdbtn,fwdbtn,False,False,False,NaN,False
3,2017-11-21T11:48:59Z,"iOS 9.3.5 (iPad2,5)",15371,BR,201.6.225.101,Gotham Projection Room,Rich Carle,Injustice 2: Original Video Game Soundtrack,spotify:track:6qNM4ljTc8PGO3hzjZj4sg,None,...,None,None,None,fwdbtn,endplay,False,False,False,NaN,False
4,2017-11-21T11:50:16Z,"iOS 9.3.5 (iPad2,5)",58653,BR,201.6.225.101,Waiting for Superman,Daughtry,Baptized (Deluxe Version),spotify:track:4AU7z13HYmPMetlWbq1mys,None,...,None,None,None,clickrow,endplay,False,False,False,NaN,False


<h3 style='color:gray;'>Imports Library JSON file, using record_path (3/4)</h3>

In [ ]:
library = import_json(PATH_LIBRARY, record_path=['tracks'])
library.head()

,artist,album,track,uri
0,Aaron Smith,Dancin (feat. Luvli),Dancin (feat. Luvli) - Krono Remix,spotify:track:6WkJ2OK163XXS2oARUC9JM
1,Duke Dumont,Ocean Drive,Ocean Drive,spotify:track:0b6wdul3A5sQNpIOv03OxP
2,Twenty One Pilots,Blurryface,Ride,spotify:track:2Z8WuEywRWYTKe1NybPQEW
3,Alvei,Summertime TikTok Trap,Summertime TikTok Trap,spotify:track:79IMx3KnX1LDuKRcO7Uwfh
4,Bastille,Bad Blood,Pompeii,spotify:track:6fNhZRFEkBfgW39W3wKARJ


<h3 style='color:gray;'>Imports Playlist JSON file, using extra method (4/4)</h3>

In [ ]:
def transform_playlist(data):
    all_playlists = []
    for i in data['playlists']:
        i_playlist = pd.json_normalize(i['items'], sep='_')
        i_playlist['playlists_lastModifiedDate'] = i['lastModifiedDate']
        i_playlist['playlists_name'] = i['name']
        all_playlists.append(i_playlist)
    return pd.concat(all_playlists)

playlist = import_json(PATH_PLAYLIST, transform=transform_playlist)
playlist.head()

,episode,audiobook,local_track,added_date,track_track_name,track_artist_name,track_album_name,track_track_uri,playlists_last_modified_date,playlists_name
0,None,None,None,2025-11-05,Ferreiro,AYAKASHI,Ferreiro,spotify:track:1bxzMBOWQG6CXwDWPCDQFh,2025-12-23,Ordem 🛡️⚜️
1,None,None,None,2025-11-05,Memórias,AYAKASHI,Memórias,spotify:track:4GJzPRzP8JrVAkF7ckecLL,2025-12-23,Ordem 🛡️⚜️
2,None,None,None,2025-11-05,Natal Macabro (Slashers),Ivou Music,Natal Macabro (Slashers),spotify:track:60QdVpCtnk4AlyiHhIkdya,2025-12-23,Ordem 🛡️⚜️
3,None,None,None,2025-11-05,Manancial,AYAKASHI,Manancial,spotify:track:0UvMzIw4CEVJzQqd440own,2025-12-23,Ordem 🛡️⚜️
4,None,None,None,2025-11-05,Abutre,AYAKASHI,Abutre,spotify:track:6NqsXRUsjFJjJu8BBrJZ10,2025-12-23,Ordem 🛡️⚜️


# Data Consolidation

In [ ]:
# 1. Merge history and marquee
final_df = pd.merge(
    history, 
    marquee, 
    how='left', 
    left_on='master_metadata_album_artist_name', 
    right_on='artist_name'
).drop(columns=['master_metadata_album_artist_name'])

# 2. Drop nulls EARLY (before applying string operations to save compute time)
final_df = final_df.dropna(subset=['spotify_track_uri'])

# 3. Extract track_id (str.split is generally more idiomatic and readable than rpartition)
final_df['track_id'] = final_df['spotify_track_uri'].str.split(':').str[-1]

# 4. Save to CSV
final_df.to_csv(PATH_SPOTIFY, index=False)

final_df.head()